# 02 - Limpieza: empleo, salarios, empresas e industria por rama

Extrae las series de los Excel de OEDE (Ministerio de Trabajo) e INDEC con las funciones de `src/loaders.py`
(cada boletín tiene un layout distinto), las lleva a la misma clave de **rama** que comercio exterior, y las
guarda en formato tidy en `data/processed/`.

## Decisiones que importan

- **"Total industria" = manufactura (CIIU 15-37)**, no toda la economía. Los archivos de OEDE traen todas las actividades.
- **Salario real por rama = promedio ponderado por puestos de trabajo** de los CIIU de 2 dígitos que la componen,
  deflactado por el IPC nacional. El IPC nacional continuo empieza en dic-2016, por lo que el salario real solo
  existe desde entonces.
- Las remuneraciones son "**por todo concepto**": incluyen el aguinaldo (junio y diciembre). Un mes suelto no es
  comparable con otro; más adelante se compara siempre con promedios de 12 meses.

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

import analysis as A
import loaders as L
from ramas import RAMA_LABELS, TOTAL

RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
PROCESSED.mkdir(exist_ok=True)

In [2]:
empleo = L.load_empleo_trimestral()
empresas = L.load_empresas_anual()
salarios = L.load_remuneraciones_mensual()
ipc = L.load_ipc_nacional()
ipi = L.load_ipi_manufacturero()
uci = L.load_capacidad_instalada()

for nombre, df in [("empleo", empleo), ("empresas", empresas), ("salarios", salarios), ("ipc", ipc), ("ipi", ipi), ("uci", uci)]:
    print(f"{nombre:9s} {df.shape}")

empleo    (2783, 6)
empresas  (667, 4)
salarios  (8625, 4)
ipc       (117, 2)
ipi       (10783, 6)
uci       (1638, 3)


## Verificaciones contra los subtotales oficiales

Si mi agrupación en ramas y mi ponderación son correctas, tienen que reproducir los agregados que las propias
fuentes publican.

In [3]:
# 1) la suma de las 16 ramas de empleo tiene que ser igual al subtotal oficial "Industria" (cuadro 4 de OEDE)
suma = empleo.groupby("periodo").puestos.sum().rename("suma_ramas")
oficial = L.load_empleo_industria_oficial().set_index("periodo").puestos_oficial
assert (suma - oficial).abs().max() == 0
print("Empleo: la suma de ramas coincide exactamente con el subtotal oficial de Industria.")

# 2) el salario nominal ponderado por puestos tiene que ser casi igual a la serie oficial de Industrias manufactureras (letra D)
pesos = empleo[["ciiu2", "periodo", "puestos"]].rename(columns={"periodo": "trim", "puestos": "peso"})
s = salarios.assign(trim=salarios["periodo"].dt.to_period("Q").dt.to_timestamp())
s = pd.merge_asof(s.sort_values("trim"), pesos.sort_values("trim"), on="trim", by="ciiu2")
ponderado = (s["salario_nominal"] * s["peso"]).groupby(s["periodo"]).sum() / s.groupby("periodo")["peso"].sum()
ratio = (ponderado / L.load_remuneracion_industria_oficial().set_index("periodo").salario_oficial).dropna()
assert ratio.between(0.98, 1.02).all()
print(f"Salarios: ponderado / oficial = {ratio.min():.3f} a {ratio.max():.3f} (n={len(ratio)} meses).")

Empleo: la suma de ramas coincide exactamente con el subtotal oficial de Industria.


Salarios: ponderado / oficial = 0.992 a 1.002 (n=363 meses).


In [4]:
E = A.empleo_por_rama(empleo)                          # + fila TOTAL
S = A.salario_real_ponderado(salarios, empleo, ipc)    # + fila TOTAL
I = A.ipi_por_rama(ipi)                                # nivel general = TOTAL
C = A.empresas_por_rama(empresas)                      # + fila TOTAL

E.to_csv(PROCESSED / "empleo_por_rama.csv", index=False)
S.to_csv(PROCESSED / "salario_real_por_rama.csv", index=False)
I.to_csv(PROCESSED / "ipi_por_rama.csv", index=False)
C.to_csv(PROCESSED / "empresas_por_rama.csv", index=False)
ipi[ipi["nivel"] == "subrama"].to_csv(PROCESSED / "ipi_subramas.csv", index=False)
uci.to_csv(PROCESSED / "capacidad_instalada.csv", index=False)
print("Guardado en data/processed/")

Guardado en data/processed/


## Vistazo: empleo total de la industria manufacturera (puestos, últimos trimestres)

In [5]:
E[E.rama == TOTAL].set_index("periodo").puestos.tail(6)

periodo
2024-10-01    1242825
2025-01-01    1249163
2025-04-01    1233637
2025-07-01    1222967
2025-10-01    1207045
2026-01-01    1200862
Name: puestos, dtype: int64